# DeepFake Detection — Google Colab Training

**Architecture:** DINOv2 ViT-B/14 + FFT/DCT LightCNN + Temperature Scaling

**Stratégie stockage (Drive limité) :**
- `data/` → `/content/` local Colab (~78 GB dispo, éphémère)
- `checkpoints/`, `outputs/`, `logs/` → Google Drive (permanent, ~1 GB max)

| Étape | Cellule |
|-------|--------|
| Vérifier GPU + espace disque | §1 |
| Monter Drive | §2 |
| Cloner repo | §3 |
| Installer dépendances | §4 |
| Lier Drive ↔ repo | §5 |
| Télécharger DF40 (subset 5 méthodes) | §6 |
| **Réorganiser DF40 + télécharger images réelles** | §6b |
| Configurer l'entraînement | §7 |
| **Entraîner** | §8 |
| Évaluer | §9 |
| Inférence | §10 |
| Afficher résultats | §11 |
| Télécharger modèle | §12 |
| Anti-idle | §13 |

## 1. Vérifier GPU et espace disque

In [ ]:
import torch, shutil

if not torch.cuda.is_available():
    raise RuntimeError("GPU non détecté !\n→ Runtime > Modifier le type d'exécution > T4 GPU")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")

total, used, free = shutil.disk_usage('/content')
print(f"\nDisque local /content/")
print(f"  Total : {total/1e9:.0f} GB")
print(f"  Libre : {free/1e9:.0f} GB")

if free / 1e9 < 15:
    print("\n⚠ Espace faible. Runtime > Déconnecter et supprimer le runtime.")
else:
    print("\nEspace suffisant.")

## 2. Monter Google Drive

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/DeepFakeDetection'

for folder in ['checkpoints', 'outputs', 'logs']:
    os.makedirs(f'{DRIVE_ROOT}/{folder}', exist_ok=True)

print(f"Drive monté : {DRIVE_ROOT}")
_, _, free = shutil.disk_usage(DRIVE_ROOT)
print(f"Drive libre : {free/1e9:.1f} GB")

## 3. Cloner / Mettre à jour le repo GitHub

In [ ]:
import os

REPO_URL = 'https://github.com/anessrb/Deepfakedetection.git'
REPO_DIR = '/content/Deepfakedetection'

if os.path.exists(f'{REPO_DIR}/.git'):
    print("Repo déjà cloné — mise à jour...")
    !cd {REPO_DIR} && git pull --quiet
else:
    print("Clonage du repo...")
    !git clone --quiet {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Répertoire : {os.getcwd()}")
!git log --oneline -3

## 4. Installer les dépendances

In [ ]:
!pip install -q -r requirements.txt

import torch, timm, albumentations, cv2
print(f"torch         : {torch.__version__}")
print(f"timm          : {timm.__version__}")
print(f"albumentations: {albumentations.__version__}")
print(f"opencv        : {cv2.__version__}")
print("Toutes les dépendances OK.")

## 5. Lier Drive ↔ Repo (symlinks)

- `data/` → `/content/data/` (local, rapide, ~78 GB)
- `checkpoints/`, `outputs/`, `logs/` → Google Drive (permanent)

In [ ]:
import os

REPO_DIR   = '/content/Deepfakedetection'
DRIVE_ROOT = '/content/drive/MyDrive/DeepFakeDetection'

def make_symlink(link, target):
    if os.path.islink(link):
        os.remove(link)
    elif os.path.isdir(link) and not os.listdir(link):
        os.rmdir(link)
    if not os.path.exists(link):
        os.symlink(target, link)
        return True
    return False

os.makedirs('/content/data', exist_ok=True)
if make_symlink(f'{REPO_DIR}/data', '/content/data'):
    print("  ✓ data/        → /content/data/ (local)")
else:
    print("  ~ data/        existe déjà")

for folder in ['checkpoints', 'outputs', 'logs']:
    target = f'{DRIVE_ROOT}/{folder}'
    if make_symlink(f'{REPO_DIR}/{folder}', target):
        print(f"  ✓ {folder:<14} → Drive (permanent)")
    else:
        print(f"  ~ {folder:<14} existe déjà")

print("\nContenu data/ :", os.listdir(f'{REPO_DIR}/data') or '(vide)')

## 6. Télécharger DF40 — subset 5 méthodes (~10 GB)

Images fakées téléchargées dans `/content/data/df40/` (local).

In [ ]:
import os
!pip install -q gdown

METHODS = {
    'simswap'  : '1vnEXjxgSxmiNY-RkLQdsbhayTvAAoOIc',
    'inswap'   : '1hEsN-oY9Ye2OiAzGn03UD7AajztZ6b_B',
    'faceswap' : '122-fpveOf2oUDwGzbhkgoVg2BrV_YVGC',
    'sadtalker': '1DQCVDlFInuAH3ryQgZIyKQzPiEIyFaaa',
    'fomm'     : '1UgGDvGGw5H6Wf0KTHzjKoigHB5ALcC5Q',
}

base = '/content/data/df40'
os.makedirs(base, exist_ok=True)

for name, file_id in METHODS.items():
    out_dir  = f'{base}/{name}'
    zip_path = f'{base}/{name}.zip'

    if os.path.isdir(out_dir) and os.listdir(out_dir):
        from pathlib import Path
        n = sum(1 for _ in Path(out_dir).rglob('*.png')) + sum(1 for _ in Path(out_dir).rglob('*.jpg'))
        print(f"  ~ {name:<12}: déjà présent ({n:,} images)")
        continue

    print(f"  ↓ {name}...")
    !gdown {file_id} -O {zip_path} --quiet

    if os.path.exists(zip_path):
        os.makedirs(out_dir, exist_ok=True)
        !unzip -q {zip_path} -d {out_dir}/
        os.remove(zip_path)
        print(f"    ✓ {name} extrait")
    else:
        print(f"    ✗ échec : {name}")

import shutil
_, _, free = shutil.disk_usage('/content')
print(f"\nEspace local restant : {free/1e9:.1f} GB")

## 6b. Réorganiser DF40 + Télécharger images réelles

Le loader attend :
```
data/df40/
├── real/        ← images de vrais visages
└── fake/
    ├── simswap/
    ├── inswap/
    └── ...
```

Cette cellule :
1. Déplace les méthodes téléchargées dans `fake/`
2. Télécharge ~25k vrais visages depuis FFHQ (HuggingFace, libre d'accès)

In [ ]:
import os, shutil
from pathlib import Path

base = Path('/content/data/df40')
fake_dir = base / 'fake'
real_dir = base / 'real'
fake_dir.mkdir(exist_ok=True)
real_dir.mkdir(exist_ok=True)

# --- Étape 1 : déplacer les méthodes dans fake/ ---
METHODS = ['simswap', 'inswap', 'faceswap', 'sadtalker', 'fomm']
print("=== Réorganisation des fakes ===")
for name in METHODS:
    src = base / name
    dst = fake_dir / name
    if src.exists() and not dst.exists():
        shutil.move(str(src), str(dst))
        n = sum(1 for _ in dst.rglob('*.png')) + sum(1 for _ in dst.rglob('*.jpg'))
        print(f"  ✓ fake/{name}: {n:,} images")
    elif dst.exists():
        n = sum(1 for _ in dst.rglob('*.png')) + sum(1 for _ in dst.rglob('*.jpg'))
        print(f"  ~ fake/{name}: déjà en place ({n:,} images)")
    else:
        print(f"  ✗ {name}: non trouvé")

# --- Étape 2 : télécharger vrais visages (FFHQ subset via HuggingFace) ---
print("\n=== Images réelles (FFHQ via HuggingFace) ===")
existing_real = list(real_dir.rglob('*.png')) + list(real_dir.rglob('*.jpg'))

if len(existing_real) >= 10000:
    print(f"  ~ Déjà présent : {len(existing_real):,} images réelles")
else:
    print("  Téléchargement FFHQ-256 subset (~25k images, ~2 GB)...")
    !pip install -q huggingface_hub
    from huggingface_hub import snapshot_download

    # FFHQ 256px subset (libre d'accès)
    try:
        snapshot_download(
            repo_id='jxie/ffhq-256',
            repo_type='dataset',
            local_dir='/content/data/ffhq_tmp',
            ignore_patterns=['*.md', '*.json', '*.parquet'],
        )
        # Déplacer les images dans real/
        for img in Path('/content/data/ffhq_tmp').rglob('*.png'):
            shutil.move(str(img), str(real_dir / img.name))
        for img in Path('/content/data/ffhq_tmp').rglob('*.jpg'):
            shutil.move(str(img), str(real_dir / img.name))
        shutil.rmtree('/content/data/ffhq_tmp', ignore_errors=True)
    except Exception as e:
        print(f"  FFHQ échoué ({e}), fallback CelebA-HQ...")
        # Fallback : CelebA-HQ subset
        snapshot_download(
            repo_id='mattmdjaga/human_parsing_dataset',
            repo_type='dataset',
            local_dir='/content/data/celeba_tmp',
            allow_patterns=['images/*'],
        )
        for img in Path('/content/data/celeba_tmp').rglob('*.jpg'):
            shutil.move(str(img), str(real_dir / img.name))
        shutil.rmtree('/content/data/celeba_tmp', ignore_errors=True)

    n_real = len(list(real_dir.rglob('*.png'))) + len(list(real_dir.rglob('*.jpg')))
    print(f"  ✓ {n_real:,} images réelles dans real/")

# --- Résumé final ---
print("\n=== Structure finale ===")
n_real = sum(1 for _ in real_dir.rglob('*.png')) + sum(1 for _ in real_dir.rglob('*.jpg'))
print(f"  real/  : {n_real:,} images")
n_fake_total = 0
for m in fake_dir.iterdir():
    if m.is_dir():
        n = sum(1 for _ in m.rglob('*.png')) + sum(1 for _ in m.rglob('*.jpg'))
        n_fake_total += n
        print(f"  fake/{m.name:<10}: {n:,} images")
print(f"  fake/  TOTAL : {n_fake_total:,} images")
print(f"  GRAND TOTAL  : {n_real + n_fake_total:,} images")

_, _, free = shutil.disk_usage('/content')
print(f"\nEspace local restant : {free/1e9:.1f} GB")

## 7. Configurer l'entraînement pour Colab

In [ ]:
import yaml, torch

with open('configs/default.yaml', 'r') as f:
    config = yaml.safe_load(f)

vram_gb    = torch.cuda.get_device_properties(0).total_memory / 1e9
batch_size = 64 if vram_gb >= 30 else 32

config['training']['batch_size']     = batch_size
config['training']['num_workers']    = 2
config['training']['epochs']         = 30
config['training']['use_amp']        = True
config['training']['checkpoint_dir'] = 'checkpoints/'
config['datasets']['data_root']      = 'data/'
config['datasets']['max_samples_per_dataset'] = None
config['evaluation']['cross_dataset_eval']    = ['df40']

with open('configs/colab.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("Config sauvegardée : configs/colab.yaml")
print(f"  GPU        : {torch.cuda.get_device_name(0)} ({vram_gb:.0f} GB VRAM)")
print(f"  batch_size : {batch_size}")
print(f"  epochs     : 30 | AMP : True")
print(f"  data_root  : data/ → /content/data/ (local)")
print(f"  checkpoints: checkpoints/ → Drive")

## 8. Entraîner le modèle

**Checkpoints → Google Drive** (survivent aux déconnexions).

In [ ]:
!python scripts/train.py \
    --config configs/colab.yaml \
    --output_dir outputs/

In [ ]:
# Reprendre après déconnexion (re-exécuter §1→§7 d'abord, §6b saute les fichiers existants)
# !python scripts/train.py \
#     --config configs/colab.yaml \
#     --resume checkpoints/best.pth \
#     --output_dir outputs/

## 9. Évaluation cross-dataset

In [ ]:
import os

for ckpt in ['outputs/detector_calibrated.pth', 'checkpoints/best.pth']:
    if os.path.exists(ckpt):
        print(f"Checkpoint : {ckpt}")
        !python scripts/evaluate.py \
            --checkpoint {ckpt} \
            --output_dir outputs/evaluation/ \
            --config configs/colab.yaml
        break
else:
    print("Aucun checkpoint. Lance §8 d'abord.")

## 10. Inférence sur image ou vidéo

In [ ]:
from google.colab import files
import os

print("Upload une image (.jpg/.png) ou vidéo (.mp4) :")
uploaded = files.upload()

ckpt = 'outputs/detector_calibrated.pth'
if not os.path.exists(ckpt):
    ckpt = 'checkpoints/best.pth'

for fname in uploaded.keys():
    print(f"\n→ Inférence : {fname}")
    !python scripts/inference.py \
        --input "{fname}" \
        --checkpoint {ckpt} \
        --output_dir outputs/inference/

## 11. Afficher les résultats

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

plots = [
    ('Training History',  'outputs/training_history.png'),
    ('Calibration Curve', 'outputs/calibration_curve.png'),
]

available = [(t, p) for t, p in plots if Path(p).exists()]
if not available:
    print("Aucun graphique. Lance l'entraînement d'abord.")
else:
    fig, axes = plt.subplots(1, len(available), figsize=(7 * len(available), 5))
    if len(available) == 1:
        axes = [axes]
    for ax, (title, path) in zip(axes, available):
        ax.imshow(mpimg.imread(path))
        ax.set_title(title, fontsize=12)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 12. Télécharger le modèle sur ton Mac

Déjà sauvegardé sur Drive. Cellule ci-dessous pour téléchargement direct :

In [ ]:
from google.colab import files
import os

for path in ['outputs/detector_calibrated.pth', 'checkpoints/best.pth']:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"Téléchargement : {path} ({size_mb:.0f} MB)")
        files.download(path)
        break
else:
    print("Aucun modèle. Lance §8 d'abord.")

## 13. Anti-idle — éviter la déconnexion Colab

Lance dans un onglet séparé pendant l'entraînement.

In [ ]:
import time
from IPython.display import clear_output

for i in range(9999):
    time.sleep(60)
    clear_output(wait=True)
    print(f"Session active — {i+1} min écoulées. (■ pour arrêter)")